In [1]:
import os
import sys

os.environ["JAVA_HOME"] = r"C:\Java17"
os.environ["HADOOP_HOME"] = r"C:\hadoop-3.5.0"
os.environ["SPARK_HOME"] = r"C:\spark-4.2.0-bin-hadoop3"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("Spark:", os.environ["SPARK_HOME"])

Python: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe
Spark: C:\spark-4.2.0-bin-hadoop3


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HotelBookingAnalysis")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 4.2.0


In [5]:
hdfs_input = (
    "hdfs://localhost:9000/"
    "hotel_booking/full_from_kafka/kafka_hotel_booking_full.jsonl"
)

df = spark.read.json(hdfs_input)

In [6]:

print("Số dòng:", df.count())
print("Số cột:", len(df.columns))

df.printSchema()
df.show(5, truncate=False)

Số dòng: 119390
Số cột: 32
root
 |-- adr: string (nullable = true)
 |-- adults: string (nullable = true)
 |-- agent: string (nullable = true)
 |-- arrival_date_day_of_month: string (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: string (nullable = true)
 |-- arrival_date_year: string (nullable = true)
 |-- assigned_room_type: string (nullable = true)
 |-- babies: string (nullable = true)
 |-- booking_changes: string (nullable = true)
 |-- children: string (nullable = true)
 |-- company: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- days_in_waiting_list: string (nullable = true)
 |-- deposit_type: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- hotel: string (nullable = true)
 |-- is_canceled: string (nullable = true)
 |-- is_repeated_guest: string (nullable = true)
 |-- lead_time: string (nullable = true)
 |-- market_segment: string (n

In [7]:
# Kiểm tra dữ liệu 
df.show(5, truncate=False)

+---+------+-----+-------------------------+------------------+------------------------+-----------------+------------------+------+---------------+--------+-------+-------+-------------+--------------------+------------+--------------------+------------+-----------+-----------------+---------+--------------+----+------------------------------+----------------------+---------------------------+------------------+-----------------------+------------------+--------------------+-----------------------+-------------------------+
|adr|adults|agent|arrival_date_day_of_month|arrival_date_month|arrival_date_week_number|arrival_date_year|assigned_room_type|babies|booking_changes|children|company|country|customer_type|days_in_waiting_list|deposit_type|distribution_channel|hotel       |is_canceled|is_repeated_guest|lead_time|market_segment|meal|previous_bookings_not_canceled|previous_cancellations|required_car_parking_spaces|reservation_status|reservation_status_date|reserved_room_type|stays_in_w

In [10]:
# kiểm tra số cột quan trọng
df.select(
    "hotel",
    "is_canceled",
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "adr",
    "country"
).show(10, truncate=False)

+------------+-----------+---------+-----------------+------------------+-----+-------+
|hotel       |is_canceled|lead_time|arrival_date_year|arrival_date_month|adr  |country|
+------------+-----------+---------+-----------------+------------------+-----+-------+
|Resort Hotel|0          |342      |2015             |July              |0    |PRT    |
|Resort Hotel|0          |737      |2015             |July              |0    |PRT    |
|Resort Hotel|0          |7        |2015             |July              |75   |GBR    |
|Resort Hotel|0          |13       |2015             |July              |75   |GBR    |
|Resort Hotel|0          |14       |2015             |July              |98   |GBR    |
|Resort Hotel|0          |14       |2015             |July              |98   |GBR    |
|Resort Hotel|0          |0        |2015             |July              |107  |PRT    |
|Resort Hotel|0          |9        |2015             |July              |103  |PRT    |
|Resort Hotel|1          |85    

In [12]:
df.printSchema()
df.describe().show()

root
 |-- adr: string (nullable = true)
 |-- adults: string (nullable = true)
 |-- agent: string (nullable = true)
 |-- arrival_date_day_of_month: string (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: string (nullable = true)
 |-- arrival_date_year: string (nullable = true)
 |-- assigned_room_type: string (nullable = true)
 |-- babies: string (nullable = true)
 |-- booking_changes: string (nullable = true)
 |-- children: string (nullable = true)
 |-- company: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- days_in_waiting_list: string (nullable = true)
 |-- deposit_type: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- hotel: string (nullable = true)
 |-- is_canceled: string (nullable = true)
 |-- is_repeated_guest: string (nullable = true)
 |-- lead_time: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- meal: 

Kiểm tra kiểu dữ liệu của các biến trong cột 

Nhận xét: nhiều biến có dữ liệu chưa phù hợp,tiến hành chuyển các dữ liệu thành dạng phù hợp hơn

In [13]:
from pyspark.sql.functions import col

df_typed = (
    df
    .withColumn("adr", col("adr").cast("double"))
    .withColumn("is_canceled", col("is_canceled").cast("int"))
    .withColumn("lead_time", col("lead_time").cast("int"))
    .withColumn("adults", col("adults").cast("int"))
    .withColumn("children", col("children").cast("double"))
    .withColumn("babies", col("babies").cast("int"))
    .withColumn("arrival_date_year", col("arrival_date_year").cast("int"))
    .withColumn("arrival_date_month", col("arrival_date_month").cast("string"))
)

df_typed.printSchema()

root
 |-- adr: double (nullable = true)
 |-- adults: integer (nullable = true)
 |-- agent: string (nullable = true)
 |-- arrival_date_day_of_month: string (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: string (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- assigned_room_type: string (nullable = true)
 |-- babies: integer (nullable = true)
 |-- booking_changes: string (nullable = true)
 |-- children: double (nullable = true)
 |-- company: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_type: string (nullable = true)
 |-- days_in_waiting_list: string (nullable = true)
 |-- deposit_type: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- is_repeated_guest: string (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- m

In [16]:
df.show(5, truncate=False)

+---+------+-----+-------------------------+------------------+------------------------+-----------------+------------------+------+---------------+--------+-------+-------+-------------+--------------------+------------+--------------------+------------+-----------+-----------------+---------+--------------+----+------------------------------+----------------------+---------------------------+------------------+-----------------------+------------------+--------------------+-----------------------+-------------------------+
|adr|adults|agent|arrival_date_day_of_month|arrival_date_month|arrival_date_week_number|arrival_date_year|assigned_room_type|babies|booking_changes|children|company|country|customer_type|days_in_waiting_list|deposit_type|distribution_channel|hotel       |is_canceled|is_repeated_guest|lead_time|market_segment|meal|previous_bookings_not_canceled|previous_cancellations|required_car_parking_spaces|reservation_status|reservation_status_date|reserved_room_type|stays_in_w

In [ ]:
# kie
df_typed.select(
    "hotel",
    "is_canceled",
    "lead_time",
    "adr"
).show(10, truncate=False)

+------------+-----------+---------+-----+
|hotel       |is_canceled|lead_time|adr  |
+------------+-----------+---------+-----+
|Resort Hotel|0          |342      |0.0  |
|Resort Hotel|0          |737      |0.0  |
|Resort Hotel|0          |7        |75.0 |
|Resort Hotel|0          |13       |75.0 |
|Resort Hotel|0          |14       |98.0 |
|Resort Hotel|0          |14       |98.0 |
|Resort Hotel|0          |0        |107.0|
|Resort Hotel|0          |9        |103.0|
|Resort Hotel|1          |85       |82.0 |
|Resort Hotel|1          |75       |105.5|
+------------+-----------+---------+-----+
only showing top 10 rows


In [18]:
#  tỉ lệ hủy đặt phòng
from pyspark.sql.functions import avg, round

df_typed.agg(
    round(avg("is_canceled") * 100, 2).alias("cancellation_rate_pct"),
    round(avg("lead_time"), 2).alias("avg_lead_time"),
    round(avg("adr"), 2).alias("avg_adr")
).show()

+---------------------+-------------+-------+
|cancellation_rate_pct|avg_lead_time|avg_adr|
+---------------------+-------------+-------+
|                37.04|       104.01| 101.83|
+---------------------+-------------+-------+

